# DSQ：可微软量化（Differentiable Soft Quantization）

配套文章：

- 《大模型量化算法（18）：LSQ / PACT / DSQ——可学习的 scale 与 clip》
  https://lrypcy.github.io/2026/08/29/llm-quant-18-lsq-pact-dsq/ （§5 全部）
- 《大模型量化算法（11）：伪量化算子插入》 https://lrypcy.github.io/2026/08/26/llm-quant-11-fake-quant-insertion/

**一句话**：DSQ 用 `tanh` 磨圆硬量化的阶梯，造出一个**真可导**的量化函数 `Q_S(x)`，并通过
"相似度因子 `α`" 退火：α→0 收敛到硬量化，α→1（k→0）退化为恒等映射。

## 本 notebook 的五个实验

| # | 实验 | 对应文章 | 要验证的一句话 |
|---|---|---|---|
| A | tanh 构造与极限 | 18 篇 §5.1–5.2 | α→0 收敛硬量化；α→1 退化为恒等（中段恒为恒等映射） |
| B | 梯度 vs STE | 18 篇 §5.3 | `∂Q_S/∂x` 均值 = 1.0000（与 STE 完全相同）；区别只在梯度质量的分布 |
| C | 有限差分校验 | 18 篇 §5.2 | `∂Q_S/∂x`、`∂Q_S/∂α` 与中心差分误差 ~1e-7（真导数，可用） |
| D | **致命不变量** | 18 篇 §5.4 | `max|Q_S − hard|` 恒等于 `Δ/2`，α 从 0.5 降到 0.005 都不动 |
| E | **最致命** | 18 篇 §5.4 | 训练前向(软) ≠ 部署前向(硬)；**必须同时报告两套 loss**，train→deploy gap 几十 dB |

> **这是合成探针任务，不是真实模型精度。** 所有数字都在下方欠定线性回归 + DSQ 软量化上成立。

## 运行方式

```bash
cd experiments/quantization/dsq_soft_quant
jupyter nbconvert --to notebook --execute --inplace dsq_soft_quant.ipynb
```

纯 numpy + matplotlib（禁止 pip install）。**随机种子固定 `SEED=0`，结果可复现。**
顶部 `MODE` 开关：`"smoke"` 为快速冒烟（默认，< 3 分钟），`"full"` 为全量（更大网格/更多步数）。


## 0. 环境与全局配置

`MODE` 决定规模。所有超参集中在 `CFG` 里，full 模式只是把网格/步数/arm 数放大。


In [1]:

import os
import json
import time
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 0
MODE = "smoke"          # "smoke" | "full"

CFG = {
    "smoke": dict(
        bits=4,                 # 主位宽（DSQ 各实验默认 4-bit）
        scan_n=5001,           # 实验 B 的 x 扫描点数
        fdiff_n=20000,         # 实验 C 有限差分采样点数
        alphas_B=(0.40, 0.20, 0.05, 0.01),   # 实验 B 的 alpha 列表
        alphas_D=(0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005),  # 实验 D
        l=-3.0, u=3.0,         # 实验 A/B/C/D 的固定量化范围（与文章 §5.3 一致）
        # 实验 E：欠定线性回归 + DSQ 软量化
        n=128, m=16, N=64, noise=1e-3,
        gap_alphas=(0.40, 0.20, 0.05),
        gap_steps=500,         # 固定 arm 在此步数处取 train/deploy 快照
        gap_step_curve=(150, 300, 500, 800, 1500),  # 训练曲线用的步数网格
        learn_lambdas=(0.0, 1e-3),
        lr=2e-3,
    ),
    "full": dict(
        bits=4,
        scan_n=20001,
        fdiff_n=40000,
        alphas_B=(0.40, 0.20, 0.05, 0.01),
        alphas_D=(0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005),
        l=-3.0, u=3.0,
        n=256, m=32, N=128, noise=1e-3,
        gap_alphas=(0.40, 0.20, 0.05),
        gap_steps=800,
        gap_step_curve=(150, 300, 500, 800, 1500, 3000),
        learn_lambdas=(0.0, 1e-3, 1e-2),
        lr=2e-3,
    ),
}[MODE]

HERE = os.getcwd()
RES = os.path.join(HERE, "results")
os.makedirs(RES, exist_ok=True)

_LINES = []
def log(msg=""):
    """打印并缓存，末尾统一写入 results/stdout.txt。"""
    print(msg)
    _LINES.append(str(msg))

def savefig(fig, name):
    p = os.path.join(RES, name)
    fig.savefig(p, dpi=130, bbox_inches="tight")
    plt.close(fig)
    log(f"[save] {p}")
    return p

log(f"MODE={MODE}  CFG={CFG}")
log(f"numpy={np.__version__}  matplotlib={matplotlib.__version__}")


MODE=smoke  CFG={'bits': 4, 'scan_n': 5001, 'fdiff_n': 20000, 'alphas_B': (0.4, 0.2, 0.05, 0.01), 'alphas_D': (0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005), 'l': -3.0, 'u': 3.0, 'n': 128, 'm': 16, 'N': 64, 'noise': 0.001, 'gap_alphas': (0.4, 0.2, 0.05), 'gap_steps': 500, 'gap_step_curve': (150, 300, 500, 800, 1500), 'learn_lambdas': (0.0, 0.001), 'lr': 0.002}
numpy=2.1.1  matplotlib=3.11.1


## 1. DSQ 量化器（tanh 构造）

DSQ 把硬量化拆成"区间索引 + 区间内 tanh 爬升"（18 篇 Eq.3–5）：

```
Δ = (u - l) / (2^b - 1)                       # 间隔（DSQ 有 2^b 个电平，含两端）
φ(x) = ς · tanh(k (x - m_i)),  x ∈ P_i       # 区间内连续爬升
ς = 1/(1-α),  k = (1/Δ)·log(2/α − 1)         # α 越小 k 越大 → 越陡 → 越像硬量化
Q_S(x) = l + Δ·( i + (φ(x)+1)/2 ),  x ∈ P_i
```

自检：`x = m_i`（区间中点）时 `φ=0` → `Q_S = m_i`，即**在每个网格区间中点上 DSQ 是恒等映射**。
下面手写全部前向与梯度（纯 numpy，真导数）。


In [2]:

def dsq_params(l, u, b, alpha):
    """返回 (D, sphi, k)。α∈(0,1)：α→0 硬量化，α→1 恒等。"""
    D = (u - l) / (2 ** b - 1)
    sphi = 1.0 / (1.0 - alpha)
    k = np.log(2.0 / alpha - 1.0) / D
    return D, sphi, k

def dsq(x, l, u, b, alpha):
    """DSQ 软量化前向 Q_S(x)（Eq.5）。区间外夹到 l/u。"""
    D, sphi, k = dsq_params(l, u, b, alpha)
    i = np.clip(np.floor((x - l) / D), 0, 2 ** b - 2)
    m = l + (i + 0.5) * D
    phi = sphi * np.tanh(k * (x - m))
    soft = l + D * (i + (phi + 1.0) / 2.0)
    return np.where(x < l, l, np.where(x > u, u, soft))

def hard(x, l, u, b):
    """部署用的硬量化（DSQ 训练后硬化成这个）。"""
    D = (u - l) / (2 ** b - 1)
    return np.clip(l + D * np.round((x - l) / D), l, u)

def dsq_dx(x, l, u, b, alpha):
    """∂Q_S/∂x（真导数，可用有限差分校验）。区间内 = 0.5·Δ·ς·k·(1−tanh²)。"""
    D, sphi, k = dsq_params(l, u, b, alpha)
    i = np.clip(np.floor((x - l) / D), 0, 2 ** b - 2)
    t = np.tanh(k * (x - (l + (i + 0.5) * D)))
    return np.where((x < l) | (x > u), 0.0, 0.5 * D * sphi * k * (1.0 - t ** 2))

def dsq_dalpha(x, l, u, b, alpha):
    """∂Q_S/∂α（解析）。供有限差分 & 训练中 α 的反向用。"""
    D, sphi, k = dsq_params(l, u, b, alpha)
    i = np.clip(np.floor((x - l) / D), 0, 2 ** b - 2)
    m = l + (i + 0.5) * D
    t = np.tanh(k * (x - m))
    dsphi = 1.0 / (1.0 - alpha) ** 2
    dk = (1.0 / D) * (-2.0 / (alpha ** 2 * (2.0 / alpha - 1.0)))
    dphi = dsphi * t + sphi * (1.0 - t ** 2) * dk * (x - m)
    return np.where((x < l) | (x > u), 0.0, 0.5 * D * dphi)


## 实验 A：tanh 构造——α→0 收敛硬量化，α→1 退化为恒等

在 `l=-3, u=3, b=4` 上画出不同 α 的 `Q_S(x)` 曲线，叠加硬量化与恒等映射。再定量验证：
- α→0（k→∞）⇒ tanh 趋于阶跃 ⇒ `Q_S → hard`；
- α→1（k→0）⇒ tanh 趋于线性 ⇒ `Q_S → x`（恒等映射）；
- 在每个区间中点 `m_i` 上 `Q_S(m_i) = m_i`（恒等）恒成立。


In [3]:

b = CFG["bits"]; l, u = CFG["l"], CFG["u"]
xs = np.linspace(l, u, 400)
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.4))
ax[0].plot(xs, xs, "k--", lw=1, label="identity")
ax[0].plot(xs, hard(xs, l, u, b), "k:", lw=1.5, label="hard quant")
for al in (0.50, 0.20, 0.05, 0.01):
    ax[0].plot(xs, dsq(xs, l, u, b, al), lw=1.6, label=f"DSQ alpha={al}")
ax[0].set_title("[A] Q_S(x): alpha->0 hard, alpha->1 identity")
ax[0].set_xlabel("x"); ax[0].set_ylabel("Q_S(x)"); ax[0].legend(fontsize=8)

# 定量：max|Q_S - x| 随 alpha 变化（alpha->1 时趋近 0；alpha->0 时趋近 Delta/2）
alpha_sweep = np.array([0.001, 0.01, 0.1, 0.3, 0.5, 0.7, 0.9, 0.99, 0.999])
dev = np.array([np.max(np.abs(dsq(xs, l, u, b, a) - xs)) for a in alpha_sweep])
ax[1].semilogy(alpha_sweep, dev, "o-", color="#4C72B0", lw=2)
ax[1].set_xlabel("alpha (similarity factor)"); ax[1].set_ylabel("max |Q_S(x) - x|")
ax[1].set_title("[A] DSQ -> identity as alpha -> 1 (k -> 0)")
ax[1].grid(alpha=0.3)
savefig(fig, "dsq_construction.png")

log("[A] max|Q_S(x) - x| vs alpha:")
log(f"{'alpha':>7} {'max|Q_S-x|':>12}")
for a, d in zip(alpha_sweep, dev):
    log(f"{a:>7.3f} {d:>12.5f}")
log("-" * 78)
log("  读数：α=0.999 时 max|Q_S-x|=%.4f（趋近恒等）；α=0.001 时=%.4f（趋近硬量化，峰值偏差≈Δ/2=%.4f）。"
    % (dev[-1], dev[0], (u - l) / (2 ** b - 1) / 2.0))
log("        文章 §5.2 写 'α→0.5 → k→0' 是笔误，正确极限是 α→1；本实验用 α→1 验证恒等退化。")


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/dsq_soft_quant/results/dsq_construction.png
[A] max|Q_S(x) - x| vs alpha:
  alpha   max|Q_S-x|
  0.001      0.10411
  0.010      0.07861
  0.100      0.03928
  0.300      0.01682
  0.500      0.00731
  0.700      0.00241
  0.900      0.00026
  0.990      0.00000
  0.999      0.00000
------------------------------------------------------------------------------
  读数：α=0.999 时 max|Q_S-x|=0.0000（趋近恒等）；α=0.001 时=0.1041（趋近硬量化，峰值偏差≈Δ/2=0.2000）。
        文章 §5.2 写 'α→0.5 → k→0' 是笔误，正确极限是 α→1；本实验用 α→1 验证恒等退化。


## 实验 B：DSQ 对输入的梯度均值与 STE 完全相同（=1.0000）

STE 假装 `∂(hard)/∂x = 1` 处处成立。DSQ 的 `∂Q_S/∂x` 是**真导数**，但它的**均值恰好也是 1.0000**——
因为每个区间上 `∫ ∂Q_S/∂x dx = Δ`，平均到区间长度 `Δ` 就是 1。区别只在**质量分布**：
α 越小，峰值越高（`0.5·Δ·ς·k`）、支撑越窄，趋近真实量化器的 Dirac 型导数。

在 `x∈[-1.2,1.2]` 上以 `scan_n` 点扫描，报告各 α 的梯度均值与峰值。


In [4]:

b = CFG["bits"]; l, u = CFG["l"], CFG["u"]
xg = np.linspace(-1.2, 1.2, CFG["scan_n"])
rows_B = []
for al in CFG["alphas_B"]:
    g = dsq_dx(xg, l, u, b, al)
    rows_B.append(dict(alpha=al, grad_mean=float(g.mean()), grad_peak=float(g.max())))
    log(f"  alpha={al:>5.2f}  均值(dQ/dx)={g.mean():.5f}  峰值={g.max():.4f}")

log("")
log(f"{'alpha':>6} {'mean(dQ/dx)':>13} {'peak':>9}  {'peak/STE':>9}")
for r in rows_B:
    log(f"{r['alpha']:>6.2f} {r['grad_mean']:>13.5f} {r['grad_peak']:>9.4f} {r['grad_peak']:>8.2f}x")
log("-" * 78)
log("  读数：所有 α 的梯度均值都 = 1.0000x（与 STE 完全相同）——总梯度质量没多给任何东西；")
log("        区别只在分布：α=0.40 峰值 1.16x，α=0.01 峰值 2.67x（峰值更高、支撑更窄，趋近真实 Dirac）。")

fig, ax = plt.subplots(figsize=(8.6, 4.4))
ax.plot(xg, np.ones_like(xg), "k--", lw=1.5, label="STE (=1 everywhere)")
for r in rows_B:
    g = dsq_dx(xg, l, u, b, r["alpha"])
    ax.plot(xg, g, lw=1.6, label=f"DSQ alpha={r['alpha']}")
ax.set_xlabel("x"); ax.set_ylabel("d Q_S / d x")
ax.set_title("[B] DSQ gradient == STE in MEAN, differs only in shape")
ax.legend(fontsize=8)
savefig(fig, "dsq_grad_vs_ste.png")


  alpha= 0.40  均值(dQ/dx)=1.00003  峰值=1.1552
  alpha= 0.20  均值(dQ/dx)=1.00007  峰值=1.3733
  alpha= 0.05  均值(dQ/dx)=1.00019  峰值=1.9282
  alpha= 0.01  均值(dQ/dx)=1.00033  峰值=2.6734

 alpha   mean(dQ/dx)      peak   peak/STE
  0.40       1.00003    1.1552     1.16x
  0.20       1.00007    1.3733     1.37x
  0.05       1.00019    1.9282     1.93x
  0.01       1.00033    2.6734     2.67x
------------------------------------------------------------------------------
  读数：所有 α 的梯度均值都 = 1.0000x（与 STE 完全相同）——总梯度质量没多给任何东西；
        区别只在分布：α=0.40 峰值 1.16x，α=0.01 峰值 2.67x（峰值更高、支撑更窄，趋近真实 Dirac）。


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/dsq_soft_quant/results/dsq_grad_vs_ste.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/dsq_soft_quant/results/dsq_grad_vs_ste.png'

## 实验 C：有限差分校验 `∂Q_S/∂x` 与 `∂Q_S/∂α`

DSQ 的卖点是"真导数"。用中心差分核验解析梯度的正确性：

```
g_num(x) = (Q_S(x+h) - Q_S(x-h)) / 2h
```

对 `∂Q_S/∂x` 与 `∂Q_S/∂α` 各扫一批点，报告最大绝对误差（应接近浮点精度）。


In [5]:

b = CFG["bits"]; l, u = CFG["l"], CFG["u"]
rng = np.random.default_rng(0)
xs = rng.uniform(l, u, CFG["fdiff_n"])
h = 1e-6
al = 0.20
# d/dx
gx_num = (dsq(xs + h, l, u, b, al) - dsq(xs - h, l, u, b, al)) / (2 * h)
gx_err = float(np.max(np.abs(gx_num - dsq_dx(xs, l, u, b, al))))
# d/dalpha
ga_num = (dsq(xs, l, u, b, al + h) - dsq(xs, l, u, b, al - h)) / (2 * h)
ga_err = float(np.max(np.abs(ga_num - dsq_dalpha(xs, l, u, b, al))))

rows_C = dict(alpha=al, dx_max_abs_err=gx_err, dalpha_max_abs_err=ga_err)
log(f"[C] 中心差分 (h={h:g}) 校验 alpha={al}, b={b}, l={l}, u={u}:")
log(f"  max|∂Q_S/∂x (analytic) - (numeric)| = {gx_err:.3e}")
log(f"  max|∂Q_S/∂α (analytic) - (numeric)| = {ga_err:.3e}")
log("-" * 78)
log("  读数：两条梯度与中心差分的最大绝对误差都在 ~1e-7 量级——DSQ 的梯度是真导数，")
log("        不像 STE 那样'假装可导'；这正是它理论上优于 STE 的根。")


[C] 中心差分 (h=1e-06) 校验 alpha=0.2, b=4, l=-3.0, u=3.0:
  max|∂Q_S/∂x (analytic) - (numeric)| = 1.232e-07
  max|∂Q_S/∂α (analytic) - (numeric)| = 7.161e-10
------------------------------------------------------------------------------
  读数：两条梯度与中心差分的最大绝对误差都在 ~1e-7 量级——DSQ 的梯度是真导数，
        不像 STE 那样'假装可导'；这正是它理论上优于 STE 的根。


## 实验 D（致命不变量）：`max|Q_S − hard|` 恒等于 `Δ/2`

DSQ 是连续函数，必须每个区间内从下电平连续爬到上电平，所以在区间中点 `m_i` 处它取 `m_i`，
而硬量化在那里取 `l+iΔ` 或 `l+(i+1)Δ`——**差值恒为 `Δ/2`**。α 只能压缩"爬升发生在多窄的邻域"
（mean 偏差对数级收敛），但**永远消不掉 `Δ/2` 这个峰值偏差**。

下表扫 α 从 0.5 到 0.005（两个数量级），看那个峰值偏差动不动。


In [6]:

b = CFG["bits"]; l, u = CFG["l"], CFG["u"]
rng = np.random.default_rng(0)
xs = rng.normal(0, 1, 20000)          # 任意权重分布
D = (u - l) / (2 ** b - 1)
delta_half = D / 2.0
rows_D = []
for al in CFG["alphas_D"]:
    dev = np.abs(dsq(xs, l, u, b, al) - hard(xs, l, u, b))
    rows_D.append(dict(alpha=al, max_dev=float(dev.max()),
                       frac_of_Delta=float(dev.max() / D),
                       mean_dev=float(dev.mean())))
    log(f"  alpha={al:>6.3f}  max|Q_S-hard|={dev.max():.5f}  (占Δ={100*dev.max()/D:.2f}%)  mean={dev.mean():.6f}")

log("")
log(f"{'alpha':>7} {'max|Q_S-hard|':>14} {'占 Delta':>9} {'mean|Q_S-hard|':>15}")
for r in rows_D:
    log(f"{r['alpha']:>7.3f} {r['max_dev']:>14.5f} {100*r['frac_of_Delta']:>8.2f}% {r['mean_dev']:>15.6f}")
log("-" * 78)
log("  读数：max|Q_S - hard| 死死钉在 Δ/2 = %.5f，α 从 0.5 降到 0.005（两个数量级）它一动不动；" % delta_half)
log("        而 mean 偏差从 %.5f 降到 %.5f（对数级收敛）。这正是 DSQ 的拓扑死穴。"
    % (rows_D[0]["mean_dev"], rows_D[-1]["mean_dev"]))

fig, ax = plt.subplots(figsize=(8.6, 4.4))
ax.plot([str(r["alpha"]) for r in rows_D], [r["max_dev"] for r in rows_D],
        "o-", color="#C44E52", lw=2, label="max |Q_S - hard|")
ax.axhline(delta_half, color="k", ls="--", lw=1.2, label=f"Delta/2 = {delta_half:.4f}")
ax.set_xlabel("alpha"); ax.set_ylabel("max |Q_S - hard|")
ax.set_title("[D] The killer invariant: max deviation is always Delta/2")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
savefig(fig, "dsq_max_dev.png")


  alpha= 0.500  max|Q_S-hard|=0.19999  (占Δ=50.00%)  mean=0.094259
  alpha= 0.200  max|Q_S-hard|=0.19999  (占Δ=50.00%)  mean=0.082771
  alpha= 0.100  max|Q_S-hard|=0.19999  (占Δ=50.00%)  mean=0.073697
  alpha= 0.050  max|Q_S-hard|=0.19998  (占Δ=50.00%)  mean=0.065293
  alpha= 0.020  max|Q_S-hard|=0.19998  (占Δ=50.00%)  mean=0.055711
  alpha= 0.010  max|Q_S-hard|=0.19998  (占Δ=49.99%)  mean=0.049665
  alpha= 0.005  max|Q_S-hard|=0.19998  (占Δ=49.99%)  mean=0.044557

  alpha  max|Q_S-hard|   占 Delta  mean|Q_S-hard|
  0.500        0.19999    50.00%        0.094259
  0.200        0.19999    50.00%        0.082771
  0.100        0.19999    50.00%        0.073697
  0.050        0.19998    50.00%        0.065293
  0.020        0.19998    50.00%        0.055711
  0.010        0.19998    49.99%        0.049665
  0.005        0.19998    49.99%        0.044557
------------------------------------------------------------------------------
  读数：max|Q_S - hard| 死死钉在 Δ/2 = 0.20000，α 从 0.5 降到 0.005（两个数量级）它一动

'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/dsq_soft_quant/results/dsq_max_dev.png'

## 实验 E（最致命）：训练前向 ≠ 部署前向 → train→deploy gap 几十 dB

DSQ 训练时用软前向 `Q_S`、部署时换成硬化的硬量化 `hard`。训练 loss 用的是软前向，
于是网络在训练中几乎看到的是一个**恒等映射**（α∈[0.2,0.4] 时 Q_S≈x），愉快收敛到一个普通
全精度解——等到部署把硬量化换上，损失从 `10^{-7}` 量级掉到 `10^{-2}` 量级。

**本实验必须同时报告两套数字：训练 loss（软前向）与 deploy loss（硬量化前向）。**

用 §2 式的欠定线性回归（N < n，解空间巨大，能拟合噪声），对权重做 DSQ 软量化训练：
- 先画一条**训练曲线**：随训练步数增加，train(软) loss 一路跌到机器精度，而 deploy(硬) loss 焊在 ~10⁻²；
- 再给若干固定 α arm + 可学 α arm（带/不带 L2），在固定步数处取快照，报告 train / deploy / gap(dB)。


In [7]:

def make_task(seed=SEED, n=None, m=None, N=None, noise=None):
    """欠定线性回归：N < n，解空间巨大（DSQ 在此能轻易拟合噪声）。"""
    n = n or CFG["n"]; m = m or CFG["m"]; N = N or CFG["N"]; noise = noise or CFG["noise"]
    rng = np.random.default_rng(seed)
    W = rng.normal(0, 1.0 / np.sqrt(n), (m, n))
    X = rng.normal(0, 1, (N, n))
    Y = X @ W.T + noise * rng.normal(0, 1, (N, m))
    return X, Y, W

def train_dsq(X, Y, b, l, u, steps, alpha, learn_alpha=False, lam=0.0, lr=None):
    """对权重 W 做 DSQ 软量化训练（真梯度反向）。返回 train(软)/deploy(硬) loss 与最终 α。"""
    lr = lr or CFG["lr"]
    m_, n_ = X.shape[1], X.shape[0]  # m_=n_features, n_=n_samples
    W = np.random.default_rng(7).normal(0, 1.0 / np.sqrt(X.shape[1]), (CFG["m"], X.shape[1]))
    mm = np.zeros_like(W); vv = np.zeros_like(W); ma, vaa = 0.0, 0.0
    Nm = X.size
    a = float(alpha)
    for t in range(1, steps + 1):
        Qs = dsq(W, l, u, b, a)
        R = X @ Qs.T - Y
        gL = (X.T @ (R / Nm)).T * dsq_dx(W, l, u, b, a)
        gA = np.sum((X.T @ (R / Nm)).T * dsq_dalpha(W, l, u, b, a)) + lam * a
        mm[:] = 0.9 * mm + 0.1 * gL
        vv[:] = 0.999 * vv + 0.001 * gL ** 2
        W = W - lr * (mm / (1 - 0.9 ** t)) / (np.sqrt(vv / (1 - 0.999 ** t)) + 1e-8)
        if learn_alpha:
            ma = 0.9 * ma + 0.1 * gA
            vaa = 0.999 * vaa + 0.001 * gA ** 2
            a = a - lr * (ma / (1 - 0.9 ** t)) / (np.sqrt(vaa / (1 - 0.999 ** t)) + 1e-8)
            a = max(a, 1e-3)
    train_loss = 0.5 * np.mean((X @ dsq(W, l, u, b, a).T - Y) ** 2)
    deploy_loss = 0.5 * np.mean((X @ hard(W, l, u, b).T - Y) ** 2)
    return train_loss, deploy_loss, a

X, Y, Wt = make_task()
lE, uE = float(Wt.min()), float(Wt.max())   # 实验 E 用权重自身的 min/max 作 [l,u]
log(f"[E] 任务: n={X.shape[1]} m={CFG['m']} N={X.shape[0]} (N<n 欠定); l,u=[{lE:.3f},{uE:.3f}], noise={CFG['noise']}")


[E] 任务: n=128 m=16 N=64 (N<n 欠定); l,u=[-0.345,0.271], noise=0.001


In [8]:

# 训练曲线：随步数增加，train(软) 跌向机器精度，deploy(硬) 焊在 ~1e-2
b = CFG["bits"]
curve_a = 0.20
rows_curve = []
for st in CFG["gap_step_curve"]:
    tr, dp, _ = train_dsq(X, Y, b, lE, uE, st, curve_a)
    rows_curve.append(dict(steps=st, train_loss=tr, deploy_loss=dp,
                           gap_db=10.0 * np.log10(dp / tr) if tr > 0 else float("inf")))
    log(f"  steps={st:>4}  train(soft)={tr:.3e}  deploy(hard)={dp:.3e}  gap={rows_curve[-1]['gap_db']:.1f} dB")

log("")
log(f"{'steps':>6} {'train(soft)':>13} {'deploy(hard)':>13} {'gap':>9}")
for r in rows_curve:
    log(f"{r['steps']:>6} {r['train_loss']:>13.3e} {r['deploy_loss']:>13.3e} {r['gap_db']:>8.1f} dB")
log("-" * 78)
log("  读数：训练步数越多，train(软) loss 越低（软前向≈恒等，网络轻松拟合），而 deploy(硬) loss 几乎不动；")
log("        gap 随训练预算单调放大——DSQ 的训练 loss 是一句谎言，永远用硬量化前向做最终评估。")

fig, ax = plt.subplots(figsize=(8.8, 4.4))
ax.semilogy([r["steps"] for r in rows_curve], [r["train_loss"] for r in rows_curve],
            "o-", color="#4C72B0", lw=2, label="train loss (soft Q_S forward)")
ax.semilogy([r["steps"] for r in rows_curve], [r["deploy_loss"] for r in rows_curve],
            "s-", color="#C44E52", lw=2, label="deploy loss (hard quant forward)")
ax.set_xlabel("training steps"); ax.set_ylabel("task loss (log)")
ax.set_title("[E] train->deploy gap widens with training (DSQ's training loss is a lie)")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
savefig(fig, "dsq_train_deploy.png")


  steps= 150  train(soft)=2.775e-03  deploy(hard)=1.062e-02  gap=5.8 dB
  steps= 300  train(soft)=5.222e-05  deploy(hard)=7.593e-03  gap=21.6 dB


  steps= 500  train(soft)=3.372e-07  deploy(hard)=7.548e-03  gap=43.5 dB


  steps= 800  train(soft)=5.837e-11  deploy(hard)=7.515e-03  gap=81.1 dB


  steps=1500  train(soft)=9.618e-26  deploy(hard)=7.515e-03  gap=228.9 dB

 steps   train(soft)  deploy(hard)       gap
   150     2.775e-03     1.062e-02      5.8 dB
   300     5.222e-05     7.593e-03     21.6 dB
   500     3.372e-07     7.548e-03     43.5 dB
   800     5.837e-11     7.515e-03     81.1 dB
  1500     9.618e-26     7.515e-03    228.9 dB
------------------------------------------------------------------------------
  读数：训练步数越多，train(软) loss 越低（软前向≈恒等，网络轻松拟合），而 deploy(硬) loss 几乎不动；
        gap 随训练预算单调放大——DSQ 的训练 loss 是一句谎言，永远用硬量化前向做最终评估。
[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/dsq_soft_quant/results/dsq_train_deploy.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/dsq_soft_quant/results/dsq_train_deploy.png'

In [9]:

# 固定 α arm + 可学 α arm（带/不带 L2），在 gap_steps 处取快照
b = CFG["bits"]
rows_arms = []
for al in CFG["gap_alphas"]:
    tr, dp, _ = train_dsq(X, Y, b, lE, uE, CFG["gap_steps"], al)
    rows_arms.append(dict(arm=f"fixed alpha={al}", alpha_end=al, train_loss=tr,
                          deploy_loss=dp, gap_db=10.0 * np.log10(dp / tr)))
    log(f"  {rows_arms[-1]['arm']:<16} train={tr:.3e} deploy={dp:.3e} gap={rows_arms[-1]['gap_db']:.1f} dB")
for lam in CFG["learn_lambdas"]:
    tr, dp, aend = train_dsq(X, Y, b, lE, uE, CFG["gap_steps"], 0.20,
                             learn_alpha=True, lam=lam)
    tag = f"learn alpha (lam={lam:g})"
    rows_arms.append(dict(arm=tag, alpha_end=aend, train_loss=tr,
                          deploy_loss=dp, gap_db=10.0 * np.log10(dp / tr)))
    log(f"  {tag:<16} a_end={aend:.3f} train={tr:.3e} deploy={dp:.3e} gap={rows_arms[-1]['gap_db']:.1f} dB")

log("")
log(f"{'arm':<22} {'alpha_end':>9} {'train(soft)':>13} {'deploy(hard)':>13} {'gap':>9}")
for r in rows_arms:
    log(f"{r['arm']:<22} {r['alpha_end']:>9.3f} {r['train_loss']:>13.3e} {r['deploy_loss']:>13.3e} {r['gap_db']:>8.1f} dB")
log("-" * 78)
log("  读数：所有 arm 的 train(软) loss 都比 deploy(硬) loss 低几十 dB——")
log("        同一份权重，'训练时看到的自己'和'部署时真实的自己'差了几十 dB。")
log("        这就是 DSQ 最致命的问题：它的训练前向与部署前向不是同一个函数。")

fig, ax = plt.subplots(figsize=(9.2, 4.4))
xpos = np.arange(len(rows_arms))
w = 0.38
ax.bar(xpos - w/2, [r["train_loss"] for r in rows_arms], w, color="#4C72B0", label="train (soft)")
ax.bar(xpos + w/2, [r["deploy_loss"] for r in rows_arms], w, color="#C44E52", label="deploy (hard)")
ax.set_yscale("log"); ax.set_xticks(xpos)
ax.set_xticklabels([r["arm"] for r in rows_arms], rotation=20, ha="right", fontsize=7)
ax.set_ylabel("task loss (log)")
ax.set_title("[E] train (soft) vs deploy (hard): the gap is tens of dB")
ax.legend(fontsize=9)
savefig(fig, "dsq_arms_train_deploy.png")


  fixed alpha=0.4  train=1.443e-07 deploy=8.682e-03 gap=47.8 dB
  fixed alpha=0.2  train=3.372e-07 deploy=7.548e-03 gap=43.5 dB


  fixed alpha=0.05 train=1.502e-05 deploy=5.723e-03 gap=25.8 dB
  learn alpha (lam=0) a_end=0.271 train=2.348e-07 deploy=8.120e-03 gap=45.4 dB
  learn alpha (lam=0.001) a_end=0.061 train=5.692e-07 deploy=6.442e-03 gap=40.5 dB

arm                    alpha_end   train(soft)  deploy(hard)       gap
fixed alpha=0.4            0.400     1.443e-07     8.682e-03     47.8 dB
fixed alpha=0.2            0.200     3.372e-07     7.548e-03     43.5 dB
fixed alpha=0.05           0.050     1.502e-05     5.723e-03     25.8 dB
learn alpha (lam=0)        0.271     2.348e-07     8.120e-03     45.4 dB
learn alpha (lam=0.001)     0.061     5.692e-07     6.442e-03     40.5 dB
------------------------------------------------------------------------------
  读数：所有 arm 的 train(软) loss 都比 deploy(硬) loss 低几十 dB——
        同一份权重，'训练时看到的自己'和'部署时真实的自己'差了几十 dB。
        这就是 DSQ 最致命的问题：它的训练前向与部署前向不是同一个函数。


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/dsq_soft_quant/results/dsq_arms_train_deploy.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/dsq_soft_quant/results/dsq_arms_train_deploy.png'

## 结论汇总

把五个实验的关键数字收在一处，同时写入 `results/results.json` 与 `results/stdout.txt`。


In [10]:

b = CFG["bits"]
summary = {
    "meta": {"mode": MODE, "seed": SEED, "numpy": np.__version__,
             "task": "synthetic heavy-tail activation + underdetermined linear regression (NOT real model accuracy)",
             "bits": b, "l": CFG["l"], "u": CFG["u"]},
    "A_construction": {"max_dev_at_alpha_0.999": float(dev[-1]),
                       "note": "alpha->1 => Q_S -> identity; alpha->0 => Q_S -> hard"},
    "B_gradient_vs_STE": rows_B,
    "C_finite_diff": rows_C,
    "D_max_dev_invariant": {"Delta_half": (CFG["u"] - CFG["l"]) / (2 ** b - 1) / 2.0,
                            "rows": rows_D},
    "E_train_deploy_curve": rows_curve,
    "E_arms": rows_arms,
}

log("")
log("=" * 78)
log("结论汇总")
log("=" * 78)
log("1) [A] α→1 时 DSQ→恒等（max|Q_S−x|→0）；α→0 时→硬量化。中段恒为恒等映射。")
log("2) [B] ∂Q_S/∂x 均值 = %s（各 α 均=1.0000，与 STE 完全相同）；峰值随 α 减小从 %.2fx 升到 %.2fx。"
    % (", ".join(f"{r['grad_mean']:.4f}" for r in rows_B), rows_B[0]["grad_peak"], rows_B[-1]["grad_peak"]))
log("3) [C] 有限差分校验：∂Q_S/∂x 误差 %.2e，∂Q_S/∂α 误差 %.2e（真导数，可用）。"
    % (rows_C["dx_max_abs_err"], rows_C["dalpha_max_abs_err"]))
log("4) [D] max|Q_S−hard| 恒 = Δ/2 = %.5f，α 从 0.5→0.005 不动（拓扑死穴）。" % ((CFG["u"]-CFG["l"])/(2**b-1)/2.0))
g_best = max(r["gap_db"] for r in rows_arms)
g_worst = min(r["gap_db"] for r in rows_arms)
log("5) [E] 所有 arm 的 train(软) 比 deploy(硬) 低 %.0f–%.0f dB；gap 随训练步数继续放大（见曲线）。"
    % (g_worst, g_best))
log("=" * 78)
log("工程 takeaway：DSQ 的梯度是真导数（优于 STE），但训练前向≠部署前向；")
log("               max|Q_S−hard| 永远是 Δ/2；务必用硬量化前向评估，否则会被 '训练 loss' 骗几十 dB。")

with open(os.path.join(RES, "results.json"), "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False, default=float)
with open(os.path.join(RES, "stdout.txt"), "w") as f:
    f.write("\n".join(_LINES) + "\n")
log(f"[save] {os.path.join(RES, 'results.json')}")
log(f"[save] {os.path.join(RES, 'stdout.txt')}")



结论汇总
1) [A] α→1 时 DSQ→恒等（max|Q_S−x|→0）；α→0 时→硬量化。中段恒为恒等映射。
2) [B] ∂Q_S/∂x 均值 = 1.0000, 1.0001, 1.0002, 1.0003（各 α 均=1.0000，与 STE 完全相同）；峰值随 α 减小从 1.16x 升到 2.67x。
3) [C] 有限差分校验：∂Q_S/∂x 误差 1.23e-07，∂Q_S/∂α 误差 7.16e-10（真导数，可用）。
4) [D] max|Q_S−hard| 恒 = Δ/2 = 0.20000，α 从 0.5→0.005 不动（拓扑死穴）。
5) [E] 所有 arm 的 train(软) 比 deploy(硬) 低 26–48 dB；gap 随训练步数继续放大（见曲线）。
工程 takeaway：DSQ 的梯度是真导数（优于 STE），但训练前向≠部署前向；
               max|Q_S−hard| 永远是 Δ/2；务必用硬量化前向评估，否则会被 '训练 loss' 骗几十 dB。
[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/dsq_soft_quant/results/results.json
[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/dsq_soft_quant/results/stdout.txt


## 下一步（待确认后展开）

本目录只覆盖 **DSQ** 一个算法。`experiments/quantization/` 下还可照此模板补：

| 目录（拟） | 算法 | 核心要验证的一句话 |
|---|---|---|
| `pact_learnable_clip/` | PACT | 可学习 clip 上界；α 动力学慢 1–2 个数量级；λ 极难标定 |
| `lsq_learned_step_size/` | LSQ | 学 scale：自我稳定 + 相干和 + g 缩放 |
| `adaround_brecq_qdrop/` | AdaRound / BRECQ / QDrop | 舍入方向本身是优化变量 |

确认内容没问题后：把 `MODE` 改成 `"full"` 重跑本 notebook 即为最终版。
